# 02 — Retrieval Verification

Memverifikasi perilaku retrieval pada 22.866 chunk sebelum menulis
`vector_store.py` dan membangun agent di atasnya.

**Kenapa notebook dulu, bukan langsung modul:**

Empat asumsi sebelumnya meleset saat diuji — coverage header (99% → 69%),
struktur dokumen (newline → spasi berulang), akurasi NER (nol dari lima
sampel benar), distribusi jalur chunking (prediksi 70/30 → aktual 96,6/3,4).

Belum ada satu pun observasi tentang apa yang keluar saat data ini
di-query. Menulis modul lebih dulu berarti mengkodekan asumsi yang
belum diuji, dan biaya memperbaikinya tumbuh seiring lapisan di atasnya.

**Tiga pertanyaan yang dijawab notebook ini:**
1. Apakah pencarian semantik menemukan kandidat yang tepat?
2. Apakah payload filter bekerja di skala 22.866 titik?
3. Apakah hybrid (filter + vector) lebih baik dari vector polos?

Pertanyaan 3 adalah klaim utama project. Sejauh ini belum diuji.

In [1]:
import os
import sys
from collections import Counter

sys.path.append("..")

from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

load_dotenv("../.env")

COLLECTION = os.getenv("QDRANT_COLLECTION", "resumes")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")

oa = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
qd = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    timeout=60,
)

info = qd.get_collection(COLLECTION)
print(f"Collection : {COLLECTION}")
print(f"Points     : {info.points_count:,}")
print(f"Dimensi    : {info.config.params.vectors.size}")

Collection : resumes
Points     : 22,866
Dimensi    : 1536


In [2]:
def embed_query(text: str) -> list[float]:
    """Embed satu query. Biaya ~$0.0000002 per query."""
    r = oa.embeddings.create(model=EMBED_MODEL, input=[text])
    return r.data[0].embedding


def search(query: str, k: int = 5, category: str = None,
           section_type: str = None) -> list:
    """Vector search dengan payload filter opsional."""
    conditions = []
    if category:
        conditions.append(
            FieldCondition(key="category", match=MatchValue(value=category))
        )
    if section_type:
        conditions.append(
            FieldCondition(key="section_type", match=MatchValue(value=section_type))
        )

    qfilter = Filter(must=conditions) if conditions else None

    return qd.query_points(
        COLLECTION,
        query=embed_query(query),
        limit=k,
        query_filter=qfilter,
    ).points


def show(hits, width: int = 160):
    """Tampilkan hasil ringkas untuk pembacaan cepat."""
    for i, h in enumerate(hits, 1):
        p = h.payload
        print(f"{i}. [{h.score:.4f}] {p['category']:22s} "
              f"{p['section_type']:14s} id={p['resume_id']}")
        print(f"   {p['text'][:width].strip()}")
        print()

### Pertanyaan 1 — Apakah pencarian semantik menemukan kandidat yang tepat?

Query domain-spesifik. Yang diamati: apakah `category` hasil teratas
sesuai domain query, dan apakah isi chunk relevan dengan pertanyaannya.

**Kriteria lolos:** minimal 4 dari 5 hasil berasal dari kategori yang
masuk akal untuk query tersebut.

In [3]:
hits = search("financial audit and compliance experience", k=5)
show(hits)

1. [0.6328] APPAREL                unknown        id=13418452
   IT COMPLIANCE AUDITOR       Career Overview     I offer 15 years' experience in various areas of the Information Technology Field. Including five years experien

2. [0.6128] ACCOUNTANT             summary        id=21338490
   I have many years of experience in accounting and finance including: audit, financial analysis, bank reconciliations, accounts payables/receivables, financial s

3. [0.6072] APPAREL                skills         id=22852364
   Information System Audit and Control Association (ISACA)  Sarbanes-Oxley  Project risk and controls   Business process review      The Institute of Internal Aud

4. [0.6032] ACCOUNTANT             experience     id=11759079
   ls at multiple audit clients, including leading the sales and inventory test work of an international company with approximately one billion in annual sales.  A

5. [0.6014] FINANCE                experience     id=27914096
   ditor   City  ,   State    

In [4]:
hits = search("kitchen management and menu development", k=5)
show(hits)

1. [0.6965] CHEF                   skills         id=16924102
   Kitchen, Bar, & Dining Room
Operations
  
Integrated Inventory Control

  Promotions & Up-selling

  Budgeting / Profit & Loss
  Management

Safety & Sanitation

2. [0.6499] CHEF                   skills         id=65373280
   cal, fresh food products to support local economies and showcase community support.  Produced or amended menus and item selections in conjunction with food and

3. [0.6359] CHEF                   experience     id=35157762
   r and Kitchen Manager       Assisted guests with making menu choices in an informative and helpful fashion.Maintained knowledge of current menu items, garnishes

4. [0.6349] CHEF                   skills         id=74522938
   t industry trends.  Developed menus, pricing and special food offerings to increase revenue and customer satisfaction.  Instructed new staff in proper food prep

5. [0.6325] CHEF                   experience     id=91268638
   Maintained updated knowledge

In [5]:
hits = search("python machine learning deployment", k=5)
show(hits)

1. [0.4131] CONSULTANT             experience     id=21156767
   ed code and corrected errors to optimize output.  Resolved customer issues by establishing workarounds and solutions to debug and create defect fixes.  Wrote us

2. [0.3966] BANKING                skills         id=34953092
   Programming Language: C/C++, Python, MATLAB, SQL, R, LUA, VBA  Machine Learning: Supervised Learning, Unsupervised Learning, Deep Neural Networks  Finance: Corp

3. [0.3881] AUTOMOBILE             experience     id=22946204
   g Logistic Regression (LASSO), Linear SVM, intersection kernel SVM and Adaboost to predict tweeter users' gender by their tweets, profiles and graphic informati

4. [0.3844] AUTOMOBILE             skills         id=18448085
   Data Science Tools: R, Base SAS, Python (Numpy, Pandas, Matplotlib, Scikit- learn), SPSS, Minitab, MATLAB, Apache Spark, SQL,
MS Excel, MS Visio, Tableau MySQL,

5. [0.3839] ENGINEERING            unknown        id=12011623
   Web scraping using Beautifu

### Temuan — kualitas pencarian semantik

_(isi setelah menjalankan)_

Yang perlu dicatat:
- Apakah kategori hasil sesuai domain query?
- Rentang skor cosine — berapa tinggi hasil teratas, berapa cepat turun?
- Apakah ada `section_type` yang mendominasi? (`header_info` berpotensi
  mendominasi karena berisi jabatan, yang mirip dengan query pencarian)

### Pertanyaan 2 — Apakah payload filter bekerja di skala nyata?

Terbukti di sandbox dengan 3 titik dan 4 dimensi. Sekarang 22.866 titik
dan 1.536 dimensi.

Diuji dua hal: filter `category` (24 nilai) dan `section_type` (8 nilai).

In [6]:
q = "audit and financial reporting"

print("=== TANPA FILTER ===")
show(search(q, k=5))

print("=== FILTER category=BANKING ===")
show(search(q, k=5, category="BANKING"))

print("=== FILTER category=CHEF (sengaja tidak relevan) ===")
show(search(q, k=5, category="CHEF"))

=== TANPA FILTER ===
1. [0.5671] FINANCE                experience     id=27914096
   ditor   City  ,   State      Performed financial statement audits for high-tech, food and beverage, financial services, health care, manufacturing and telecommu

2. [0.5450] ACCOUNTANT             skills         id=29821051
   ve accounting operations procedures.  Supervised and coordinated projects for external auditors and examiner evaluations.  Articulated audit findings, risks and

3. [0.5413] CONSULTANT             skills         id=39308779
   accounting, accounting software, audit reports, audit report, audit reporting, consulting, Dell, filing, finance, financial, financial reporting, Focus, HP, IBM

4. [0.5373] ACCOUNTANT             summary        id=78257294
   Settlements Internal & External Audits SOX Compliance System Implementation & Optimization P&L Analysis & Reporting Gathering, Processing & Analyzing Data Inter

5. [0.5373] FINANCE                header_info    id=26530575
   asures

### Temuan — payload filtering

_(isi setelah menjalankan)_

Yang menentukan: apakah filter CHEF benar-benar hanya mengembalikan
chunk CHEF, meski skornya jauh lebih rendah? Itu bukti metadata
mengalahkan skor vektor — mekanisme yang sama dengan uji sandbox,
sekarang di skala penuh.

### Pertanyaan 3 — Apakah hybrid lebih baik dari vector polos?

**Ini klaim utama project.** Kalau filter tidak memberi perbaikan
terukur, hybrid retrieval hanya menambah kompleksitas tanpa manfaat —
dan lebih jujur untuk tidak memakainya.

**Metode:** query yang menyebut kategori spesifik. Hitung berapa dari
top-5 yang benar-benar dari kategori itu, dengan dan tanpa filter.

**Kriteria:** filter dianggap berguna kalau precision naik signifikan,
bukan sekadar berbeda.

In [7]:
cases = [
    ("experienced banking professional with risk management", "BANKING"),
    ("registered nurse patient care experience", "HEALTHCARE"),
    ("civil construction project supervision", "CONSTRUCTION"),
    ("digital marketing campaign management", "DIGITAL-MEDIA"),
]

print(f"{'query':45s} {'no filter':>12s} {'with filter':>12s}")
print("-" * 72)

for q, cat in cases:
    plain = search(q, k=5)
    filtered = search(q, k=5, category=cat)

    p_hit = sum(1 for h in plain if h.payload["category"] == cat)
    f_hit = sum(1 for h in filtered if h.payload["category"] == cat)

    print(f"{q[:43]:45s} {p_hit}/5{'':>9s} {f_hit}/5")

query                                            no filter  with filter
------------------------------------------------------------------------
experienced banking professional with risk    3/5          5/5
registered nurse patient care experience      2/5          5/5
civil construction project supervision        4/5          5/5
digital marketing campaign management         4/5          5/5


In [8]:
# Distribusi kategori pada hasil tanpa filter — seberapa menyebar?
for q, cat in cases:
    hits = search(q, k=10)
    dist = Counter(h.payload["category"] for h in hits)
    print(f"\n{q[:50]}")
    print(f"  target: {cat}")
    print(f"  hasil : {dict(dist)}")


experienced banking professional with risk managem
  target: BANKING
  hasil : {'BANKING': 7, 'BUSINESS-DEVELOPMENT': 2, 'FINANCE': 1}

registered nurse patient care experience
  target: HEALTHCARE
  hasil : {'AGRICULTURE': 1, 'HEALTHCARE': 5, 'ADVOCATE': 3, 'FITNESS': 1}

civil construction project supervision
  target: CONSTRUCTION
  hasil : {'CONSTRUCTION': 8, 'SALES': 1, 'FINANCE': 1}

digital marketing campaign management
  target: DIGITAL-MEDIA
  hasil : {'SALES': 2, 'DIGITAL-MEDIA': 5, 'HEALTHCARE': 1, 'CONSULTANT': 1, 'PUBLIC-RELATIONS': 1}


### Temuan — nilai hybrid retrieval

_(isi setelah menjalankan)_

Kalau vector polos sudah mengembalikan 4-5/5 dari kategori yang benar,
filter tidak menambah banyak — dan itu temuan yang harus dilaporkan
jujur, bukan disembunyikan.

Kalau vector polos menyebar ke banyak kategori, filter terbukti perlu.

### Apakah section filter berguna?

Query berorientasi pengalaman kerja seharusnya lebih baik dengan
`section_type="experience"` — chunk Skills berisi daftar keyword tanpa
konteks, chunk Education tidak relevan untuk pertanyaan pengalaman.

In [9]:
q = "managed a team of 20 people across multiple departments"

print("=== TANPA section filter ===")
show(search(q, k=5), width=140)

print("=== section_type=experience ===")
show(search(q, k=5, section_type="experience"), width=140)

=== TANPA section filter ===
1. [0.5914] ADVOCATE               accomplishments id=36392131
   Coordinated all department functions for team of 10+ employees.Received a merit raise for strong attention to detail, exemplary customer ser

2. [0.5765] ENGINEERING            header_info    id=10624813
   any teams and companies to become smooth running agile groups, drastically reducing delivery issues, making the work very transparent, empow

3. [0.5744] AUTOMOBILE             accomplishments id=97449528
   General      Coordinated all department functions for team of 10+ employees.  Received a merit raise for strong attention to detail, exempla

4. [0.5613] HEALTHCARE             skills         id=23110214
   such as new facility acquisitions and business unit expansions.  Managed staff and budget 50 direct reports and up to 150 indirect reports.

5. [0.5537] FITNESS                accomplishments id=76530505
   Increased office organization by developing more efficient filing system and

In [10]:
# Distribusi section_type pada hasil tanpa filter
probe = [
    "managed a team across departments",
    "proficient in Excel and SAP",
    "bachelor degree in business administration",
]

for q in probe:
    hits = search(q, k=10)
    dist = Counter(h.payload["section_type"] for h in hits)
    print(f"\n{q}")
    print(f"  {dict(dist)}")


managed a team across departments
  {'header_info': 1, 'experience': 4, 'accomplishments': 3, 'skills': 1, 'summary': 1}

proficient in Excel and SAP
  {'skills': 10}

bachelor degree in business administration
  {'education': 10}


### Temuan — section filtering

_(isi setelah menjalankan)_

## Kesimpulan

_(isi setelah semua cell dijalankan)_

| Pertanyaan | Terjawab? | Temuan |
|---|---|---|
| Pencarian semantik akurat? | | |
| Payload filter bekerja di skala penuh? | | |
| Hybrid lebih baik dari vector polos? | | |
| Section filter berguna? | | |

**Keputusan untuk `vector_store.py`:**

_(isi setelah menjalankan)_

**Yang perlu dikoreksi di guideline:**

_(isi setelah menjalankan)_